# TASK 3: Cleaning Data

## Objective

The objective of this project is to demonstrate professional-level data cleaning skills by transforming a deliberately messy crime incidents dataset into a clean, consistent, and analysis-ready dataset.

The cleaning process includes identifying and handling missing values, removing duplicate records, standardizing inconsistent formats, detecting and treating outliers, correcting data types, and validating the overall data quality.

Every major cleaning decision is documented and justified to ensure that the data preparation process is transparent and reproducible.

## Tech Stack

- Python
- Pandas
- NumPy
- Jupyter Notebook

## 1. Import Libraries

The required Python libraries are imported for data manipulation, numerical operations, and data quality analysis.

In [1]:
import pandas as pd
import numpy as np

## 2. Load the Dataset

The original messy crime incidents dataset is loaded using Pandas. The original dataset is preserved separately so that data quality can be compared before and after cleaning.

In [2]:
df = pd.read_csv("D:\Vamshi's Github\DataAnalytics-L1-CleaningData\DataAnalytics-L1-CleaningData\Data\crime_incidents_messy.csv")

# Preserve a copy of the original dataset
df_original = df.copy()

df.head()

,incident_id,crime_type,district,city,state,address,latitude,longitude,incident_datetime,officer_id,...,victim_gender,victim_phone,weapon_used,severity,case_status,resolution,num_arrests,property_loss_usd,reported_online,notes
0,INC001115,asslt,Sou,Maplewood,GA,2830 Cedar Lane,170.284100,-77.500710,2024-04-16 08:45:03,OFF0078,...,Unknown,6223265920,Firearm,2,Open,No Arrest,NaN,NaN,True,Incident at 2830 Cedar Lane. Officer responded...
1,INC004706,burglary,southeast,Maplewood,OH,2361 Park Rd,29.422717,-77.167016,2022-01-11 22:03:29,OFF0059,...,NaN,241-973-4826,Firearm,3,NaN,NaN,2.0,44839.49,yes,Incident at 2361 Park Rd. Officer responded at...
2,INC002249,Homocide,Southwest,Lakewood,AZ,1067 Main St,39.411460,-98.615298,2022-04-14 11:39:24,OFF0088,...,Other,244-584-7696,KNIFE,1,CLOSED,warning,2.0,15963.38,YES,Incident at 1067 Main St. Officer responded at...
3,INC000021,Property Damage,Cen,Springfield,AZ,4713 Washington Ave,28.633439,-99.030051,2020-12-09 15:14:24,OFF0077,...,NaN,5021227484,hands,Low,Resolved,NaN,5.0,48680.14,False,Incident at 4713 Washington Ave. Officer respo...
4,INC000488,Domestc Violence,North,Lakewood,PA,1371 River Rd,34.217850,-121.614731,2021-07-24 20:08:13,OFF0083,...,M,9698766873,Unarmed,MEDIUM,Closed,Warning Issued,2.0,23513.01,YES,NaN


In [3]:
print("Dataset Shape:", df.shape)
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

df.info()

Dataset Shape: (5250, 33)
Number of Rows: 5250
Number of Columns: 33
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5250 entries, 0 to 5249
Data columns (total 33 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   incident_id         5250 non-null   object 
 1   crime_type          5250 non-null   object 
 2   district            5250 non-null   object 
 3   city                5250 non-null   object 
 4   state               5250 non-null   object 
 5   address             5250 non-null   object 
 6   latitude            4992 non-null   float64
 7   longitude           4961 non-null   float64
 8   incident_datetime   4910 non-null   object 
 9   officer_id          5250 non-null   object 
 10  officer_first_name  5250 non-null   object 
 11  officer_last_name   5250 non-null   object 
 12  badge_number        4938 non-null   float64
 13  suspect_id          4440 non-null   object 
 14  suspect_first_name  4440 non-null  

## 3. Initial Data Quality Report

Before performing any cleaning operations, the dataset is systematically assessed to identify potential data quality issues.

The assessment includes:

- Missing values in each column
- Duplicate records
- Current data types and potential type inconsistencies
- Unique values in categorical columns
- Numeric value ranges and possible anomalies

This initial assessment provides a baseline for comparing data quality before and after the cleaning process.

In [4]:
# Missing value report
missing_report = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_report = missing_report[missing_report['Missing_Count'] > 0]
missing_report.sort_values(by='Missing_Count', ascending=False)

,Missing_Count,Missing_Percentage
suspect_race,1601,30.50
suspect_gender,1413,26.91
suspect_age,1097,20.90
notes,1069,20.36
victim_phone,1056,20.11
victim_gender,1000,19.05
weapon_used,970,18.48
resolution,951,18.11
suspect_first_name,810,15.43
suspect_id,810,15.43


### 3.2 Duplicate Records

Duplicate records can lead to biased analysis and incorrect statistical results. Therefore, the dataset is checked for completely duplicated rows before cleaning.

The number of duplicate records is recorded so that the effect of duplicate removal can be documented later.

In [5]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()

print("Total Rows:", len(df))
print("Duplicate Rows:", duplicate_count)
print("Duplicate Percentage:", round((duplicate_count / len(df)) * 100, 2), "%")

Total Rows: 5250
Duplicate Rows: 200
Duplicate Percentage: 3.81 %


In [6]:
# Display sample duplicate records
df[df.duplicated(keep=False)].sort_values(by='incident_id').head(10)

,incident_id,crime_type,district,city,state,address,latitude,longitude,incident_datetime,officer_id,...,victim_gender,victim_phone,weapon_used,severity,case_status,resolution,num_arrests,property_loss_usd,reported_online,notes
490,INC000008,CYBER CRIME,Wes,Hillcrest,TX,5795 Washington Ave,45.335098,-100.868376,2023-04-13 07:52:34,OFF0040,...,m,8859845065,hands,High,Investgation,No Arrest,3.0,43608.34,0,Incident at 5795 Washington Ave. Officer respo...
858,INC000008,CYBER CRIME,Wes,Hillcrest,TX,5795 Washington Ave,45.335098,-100.868376,2023-04-13 07:52:34,OFF0040,...,m,8859845065,hands,High,Investgation,No Arrest,3.0,43608.34,0,Incident at 5795 Washington Ave. Officer respo...
4598,INC000015,asslt,WEST,Riverside,GA,3034 Park Rd,29.976847,-96.821673,2024-03-01 04:58:24,OFF0135,...,female,914-229-9042,Hands/Feet,3,Open,Arres Made,2.0,23469.85,0,Incident at 3034 Park Rd. Officer responded at...
37,INC000015,asslt,WEST,Riverside,GA,3034 Park Rd,29.976847,-96.821673,2024-03-01 04:58:24,OFF0135,...,female,914-229-9042,Hands/Feet,3,Open,Arres Made,2.0,23469.85,0,Incident at 3034 Park Rd. Officer responded at...
1604,INC000016,Scam,SOUTHEAST,Oakdale,IL,9317 Lincoln St,26.385297,-72.074344,2018-11-05 07:23:28,OFF0054,...,F,6618058699,Gun,Low,Pending,NaN,-5.0,14135.92,0,NaN
1564,INC000016,Scam,SOUTHEAST,Oakdale,IL,9317 Lincoln St,26.385297,-72.074344,2018-11-05 07:23:28,OFF0054,...,F,6618058699,Gun,Low,Pending,NaN,-5.0,14135.92,0,NaN
2073,INC000019,Hacking,Nor,Lakewood,TX,4527 River Rd,32.817131,-81.038695,2020-04-10 01:50:14,OFF0001,...,F,NaN,blunt object,Crit,open,Case Dismissed,3.0,2182.36,0,Incident at 4527 River Rd. Officer responded a...
2567,INC000019,Hacking,Nor,Lakewood,TX,4527 River Rd,32.817131,-81.038695,2020-04-10 01:50:14,OFF0001,...,F,NaN,blunt object,Crit,open,Case Dismissed,3.0,2182.36,0,Incident at 4527 River Rd. Officer responded a...
188,INC000024,dui,Sou,Springfield,FL,244 Maple Dr,43.169647,-109.627556,2020-10-02 22:09:01,OFF0005,...,MALE,7153261710,Gun,CRITICAL,NaN,Arrest Made,0.0,49929.34,1,Incident at 244 Maple Dr. Officer responded at...
2394,INC000024,dui,Sou,Springfield,FL,244 Maple Dr,43.169647,-109.627556,2020-10-02 22:09:01,OFF0005,...,MALE,7153261710,Gun,CRITICAL,NaN,Arrest Made,0.0,49929.34,1,Incident at 244 Maple Dr. Officer responded at...


### 3.3 Data Type Assessment

The current data types are examined to identify columns that may have been loaded with inappropriate types.

Special attention is given to date/time fields, identifiers, monetary values, and Boolean fields because these columns often require explicit type conversion during data cleaning.

In [7]:
# Display current data types
dtype_report = pd.DataFrame({
    'Current_Dtype': df.dtypes.astype(str)
})

dtype_report

,Current_Dtype
incident_id,object
crime_type,object
district,object
city,object
state,object
address,object
latitude,float64
longitude,float64
incident_datetime,object
officer_id,object


In [8]:
# Inspect sample values from columns with potential data type issues
columns_to_inspect = [
    'incident_datetime',
    'badge_number',
    'property_loss_usd',
    'reported_online'
]

for column in columns_to_inspect:
    print(f"\n--- {column} ---")
    print("Current dtype:", df[column].dtype)
    print("Sample values:", df[column].dropna().unique()[:10])


--- incident_datetime ---
Current dtype: object
Sample values: ['2024-04-16 08:45:03' '2022-01-11 22:03:29' '2022-04-14 11:39:24'
 '2020-12-09 15:14:24' '2021-07-24 20:08:13' '05-03-2020'
 '2021-10-03 05:17:02' '2023-09-30 14:49:01' '2021-01-22 02:37:24'
 '2021-03-29 04:16:18']

--- badge_number ---
Current dtype: float64
Sample values: [1342. 5133. 7781. 5097. 6220. 7082. 7747. 7869. 8229. 7242.]

--- property_loss_usd ---
Current dtype: object
Sample values: ['44839.49' '15963.38' '48680.14' '23513.01' '19075.36' '34452.47'
 '35339.52' '27438.67' '41738.8' '-34291.84']

--- reported_online ---
Current dtype: object
Sample values: ['True' 'yes' 'YES' 'False' 'no' 'Yes' 'No' '1' 'NO' '0']


### 3.4 Value Range and Anomaly Assessment

Numeric columns are examined for minimum and maximum values to identify values that fall outside logically valid ranges.

At this stage, suspicious values are only identified and documented. Their treatment will be decided during the cleaning and outlier-handling stages.

In [9]:
# Summary statistics for numeric columns
df.describe().T

,count,mean,std,min,25%,50%,75%,max
latitude,4992.0,40.778811,23.243225,25.002420,30.966472,36.967123,43.051502,199.9842
longitude,4961.0,-88.607610,37.214207,-121.975849,-108.040512,-94.656440,-80.588056,99.7873
badge_number,4938.0,5558.581612,2591.581049,1002.000000,3316.250000,5567.000000,7862.000000,9999.0000
suspect_age,4153.0,48.418493,43.504819,-75.000000,29.000000,46.000000,62.000000,298.0000
victim_age,4688.0,52.349403,48.104315,-90.000000,27.750000,49.000000,71.000000,298.0000
num_arrests,4917.0,2.262152,1.994556,-5.000000,1.000000,2.000000,4.000000,5.0000


In [10]:
# Check values outside logically valid ranges

range_anomalies = {
    'Invalid Latitude': ((df['latitude'] < -90) | (df['latitude'] > 90)).sum(),
    'Invalid Longitude': ((df['longitude'] < -180) | (df['longitude'] > 180)).sum(),
    'Invalid Suspect Age': ((df['suspect_age'] < 0) | (df['suspect_age'] > 120)).sum(),
    'Invalid Victim Age': ((df['victim_age'] < 0) | (df['victim_age'] > 120)).sum(),
    'Negative Arrest Count': (df['num_arrests'] < 0).sum()
}

pd.Series(range_anomalies, name='Anomaly_Count')

Invalid Latitude         184
Invalid Longitude          0
Invalid Suspect Age      333
Invalid Victim Age       408
Negative Arrest Count    184
Name: Anomaly_Count, dtype: int64

### 3.5 Categorical Value Inconsistencies

Categorical columns are inspected for inconsistent capitalization, abbreviations, spelling variations, and alternative representations of the same category.

Identifying these inconsistencies before cleaning helps determine the appropriate standardization rules for each column.

In [11]:
# Inspect unique values in important categorical columns
categorical_columns = [
    'crime_type',
    'district',
    'suspect_gender',
    'victim_gender',
    'weapon_used',
    'severity',
    'case_status',
    'resolution',
    'reported_online'
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False))


--- crime_type ---
crime_type
Fire Setting         84
DV                   82
ARSON                80
Drug Offence         79
Deception            79
                     ..
  Sex Assault         1
assault & battery     1
CYBER  CRIME          1
  dui                 1
  theft/larceny       1
Name: count, Length: 182, dtype: int64

--- district ---
district
Nor               273
Sou               265
SOUTHWEST         106
South             102
NORTH             101
                 ... 
   Southeast        1
  West              1
southeast           1
mid                 1
  northeast         1
Name: count, Length: 131, dtype: int64

--- suspect_gender ---
suspect_gender
NaN        1413
F           361
female      347
male        326
Unknown     323
f           323
M           317
Other       315
MALE        311
Female      309
FEMALE      305
m           301
Male        299
Name: count, dtype: int64

--- victim_gender ---
victim_gender
NaN        1000
Male        394
Unknown     383


## 4. Duplicate Removal

The initial data quality assessment identified 200 completely duplicated rows, representing approximately 3.81% of the dataset.

Since these rows are exact duplicates and do not represent additional unique incidents, they are removed to prevent repeated observations from affecting subsequent analysis.

The first occurrence of each record is retained.

In [12]:
# Record the number of rows before duplicate removal
rows_before = len(df)

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Calculate number of removed duplicates
duplicates_removed = rows_before - len(df)

print("Rows Before:", rows_before)
print("Duplicates Removed:", duplicates_removed)
print("Rows After:", len(df))

Rows Before: 5250
Duplicates Removed: 200
Rows After: 5050


In [13]:
# Verify that no exact duplicate rows remain
print("Remaining Duplicate Rows:", df.duplicated().sum())

Remaining Duplicate Rows: 0


## 5. Missing Data Handling

Missing values are handled according to the meaning, data type, and distribution of each column rather than applying a single strategy to the entire dataset.

Numerical columns are evaluated for appropriate statistical imputation, categorical columns are considered for mode or meaningful placeholder values, and records are removed only when missing information makes them unsuitable for analysis.

Each treatment decision is documented before implementation.

In [14]:
# Missing values after duplicate removal
missing_after_duplicates = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_after_duplicates = missing_after_duplicates[
    missing_after_duplicates['Missing_Count'] > 0
]

missing_after_duplicates.sort_values(
    by='Missing_Count',
    ascending=False
)

,Missing_Count,Missing_Percentage
suspect_race,1525,30.20
suspect_gender,1352,26.77
suspect_age,1052,20.83
notes,1028,20.36
victim_phone,1019,20.18
victim_gender,963,19.07
weapon_used,930,18.42
resolution,912,18.06
suspect_first_name,775,15.35
suspect_id,775,15.35


In [15]:
# Inspect numeric columns with missing values
numeric_missing_columns = [
    'latitude',
    'longitude',
    'badge_number',
    'suspect_age',
    'victim_age',
    'num_arrests'
]

df[numeric_missing_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
latitude,4799.0,40.740995,23.192947,25.002420,30.956084,36.974169,42.994098,199.9842
longitude,4772.0,-88.751699,37.035085,-121.975849,-108.078417,-94.668290,-80.886362,99.7873
badge_number,4749.0,5554.343441,2594.152677,1002.000000,3305.000000,5567.000000,7861.000000,9999.0000
suspect_age,3998.0,48.560030,43.695769,-75.000000,29.000000,46.000000,62.000000,298.0000
victim_age,4508.0,52.228039,47.832059,-90.000000,27.000000,49.000000,71.000000,298.0000
num_arrests,4726.0,2.264706,1.992146,-5.000000,1.000000,2.000000,4.000000,5.0000


### 5.2 Handling Invalid Numeric Values

Before performing missing-value imputation, numeric columns were checked for logically impossible values.

The following domain rules were applied:

- Latitude must be between -90 and 90 degrees.
- Suspect and victim ages must be between 0 and 120 years.
- The number of arrests cannot be negative.
- Longitude values were retained because the observed values fall within the valid range of -180 to 180 degrees.

Values violating these rules were treated as invalid data and replaced with missing values (`NaN`). This prevents invalid observations from influencing subsequent statistical imputation.

In [16]:
# Convert logically invalid numeric values to NaN

df.loc[~df['latitude'].between(-90, 90), 'latitude'] = np.nan

df.loc[~df['suspect_age'].between(0, 120), 'suspect_age'] = np.nan

df.loc[~df['victim_age'].between(0, 120), 'victim_age'] = np.nan

df.loc[df['num_arrests'] < 0, 'num_arrests'] = np.nan

In [17]:
# Verify that invalid numeric values have been removed

invalid_after = {
    'Invalid Latitude': ((df['latitude'] < -90) | (df['latitude'] > 90)).sum(),
    'Invalid Suspect Age': ((df['suspect_age'] < 0) | (df['suspect_age'] > 120)).sum(),
    'Invalid Victim Age': ((df['victim_age'] < 0) | (df['victim_age'] > 120)).sum(),
    'Negative Arrest Count': (df['num_arrests'] < 0).sum()
}

pd.Series(invalid_after, name='Remaining_Invalid_Values')

Invalid Latitude         0
Invalid Suspect Age      0
Invalid Victim Age       0
Negative Arrest Count    0
Name: Remaining_Invalid_Values, dtype: int64

In [18]:
# Recheck missing values after converting invalid values to NaN
missing_before_imputation = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_before_imputation = missing_before_imputation[
    missing_before_imputation['Missing_Count'] > 0
]

missing_before_imputation.sort_values(
    'Missing_Count',
    ascending=False
)

,Missing_Count,Missing_Percentage
suspect_race,1525,30.20
suspect_age,1375,27.23
suspect_gender,1352,26.77
notes,1028,20.36
victim_phone,1019,20.18
victim_gender,963,19.07
victim_age,931,18.44
weapon_used,930,18.42
resolution,912,18.06
suspect_first_name,775,15.35


In [19]:
# Check numeric distributions after invalid-value treatment
df[
    ['latitude', 'longitude', 'suspect_age',
     'victim_age', 'num_arrests', 'badge_number']
].describe().T

,count,mean,std,min,25%,50%,75%,max
latitude,4623.0,36.530899,6.663805,25.002420,30.778909,36.549199,42.206334,47.999006
longitude,4772.0,-88.751699,37.035085,-121.975849,-108.078417,-94.668290,-80.886362,99.787300
suspect_age,3675.0,45.429932,17.449443,15.000000,31.000000,46.000000,60.000000,75.000000
victim_age,4119.0,49.881525,23.309186,10.000000,30.000000,50.000000,70.000000,90.000000
num_arrests,4551.0,2.464733,1.720471,0.000000,1.000000,2.000000,4.000000,5.000000
badge_number,4749.0,5554.343441,2594.152677,1002.000000,3305.000000,5567.000000,7861.000000,9999.000000


### 5.3 Missing-Value Treatment Strategy

Missing values are handled according to the meaning and characteristics of each feature rather than using a single imputation technique for the entire dataset.

The following strategies are used:

- **Median imputation** is used for `suspect_age` and `victim_age` because age is numerical and the median is robust to extreme values.
- **Median imputation** is used for `latitude` and `longitude` to preserve records with missing geographic coordinates without allowing extreme observations to strongly influence the imputed value.
- **Mode imputation** is used for `num_arrests`, `severity`, `case_status`, `resolution`, `weapon_used`, and `reported_online`, where the most frequently occurring valid category/value provides a reasonable replacement.
- Missing `suspect_gender` and `suspect_race` values are assigned `Unknown` because the absence of demographic information should not be interpreted as the most common demographic group.
- Missing suspect identifiers and names are assigned `Unknown` because inventing an ID or person name would be misleading.
- Missing victim identifiers, names, and phone numbers are assigned `Unknown` for the same reason.
- Missing `notes` values are assigned `No Notes` because the absence of notes represents unavailable descriptive information rather than a statistical quantity.
- Missing `badge_number` values are assigned `Unknown` because badge numbers are identifiers and should not be statistically imputed.
- Rows with a missing `incident_datetime` are removed because the incident timestamp is an important event-level field and cannot be reliably reconstructed from other attributes.

In [20]:
# Median imputation for numerical columns
for column in ['suspect_age', 'victim_age', 'latitude', 'longitude']:
    df[column] = df[column].fillna(df[column].median())

# Mode imputation
mode_columns = [
    'num_arrests',
    'severity',
    'case_status',
    'resolution',
    'weapon_used',
    'reported_online'
]

for column in mode_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

# Meaningful placeholders for unavailable categorical information
unknown_columns = [
    'suspect_gender',
    'suspect_race',
    'suspect_id',
    'suspect_first_name',
    'suspect_last_name',
    'victim_id',
    'victim_first_name',
    'victim_last_name',
    'victim_phone'
]

for column in unknown_columns:
    df[column] = df[column].fillna('Unknown')

# Missing descriptive notes
df['notes'] = df['notes'].fillna('No Notes')

# Badge number is an identifier, so do not use statistical imputation
df['badge_number'] = df['badge_number'].fillna('Unknown')

# Property loss will be handled after numeric conversion

In [21]:
missing_dates = df['incident_datetime'].isnull().sum()
print("Rows with missing incident datetime:", missing_dates)

Rows with missing incident datetime: 329


In [22]:
# Remove records where the incident date/time is unavailable
df = df.dropna(subset=['incident_datetime']).copy()

print("Rows remaining:", len(df))

Rows remaining: 4721


In [23]:
pd.to_numeric(df['property_loss_usd'], errors='coerce')

0            NaN
1       44839.49
2       15963.38
3       48680.14
4       23513.01
          ...   
5245    44417.98
5246    16330.48
5247    13138.25
5248    43851.12
5249    38856.84
Name: property_loss_usd, Length: 4721, dtype: float64

In [24]:
print("Rows remaining:", len(df))
print("\nRemaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Rows remaining: 4721

Remaining missing values:
victim_gender        895
property_loss_usd    388
dtype: int64


In [25]:
# Preserve missing victim gender as an explicit unknown category
df['victim_gender'] = df['victim_gender'].fillna('Unknown')

In [26]:
# Convert property loss to numeric
df['property_loss_usd'] = pd.to_numeric(
    df['property_loss_usd'],
    errors='coerce'
)

print("Data type:", df['property_loss_usd'].dtype)
print("Missing values:", df['property_loss_usd'].isnull().sum())
print(df['property_loss_usd'].describe())

Data type: float64
Missing values: 521
count     4200.000000
mean     22755.510845
std      17582.990149
min     -49866.740000
25%      10780.970000
50%      23566.445000
75%      36781.747500
max      49998.300000
Name: property_loss_usd, dtype: float64


In [27]:
# Identify invalid negative monetary values
negative_property_loss = (df['property_loss_usd'] < 0).sum()

print("Negative property loss values:", negative_property_loss)

df.loc[df['property_loss_usd'] < 0, 'property_loss_usd'] = np.nan

Negative property loss values: 180


In [28]:
# Impute missing/invalid property loss using the median
property_loss_median = df['property_loss_usd'].median()

df['property_loss_usd'] = df['property_loss_usd'].fillna(
    property_loss_median
)

print("Property Loss Median Used:", property_loss_median)

Property Loss Median Used: 24656.765


In [29]:
# Verify remaining missing values
remaining_missing = df.isnull().sum()

print("Total Missing Values:", remaining_missing.sum())
print("\nColumns with Missing Values:")
print(remaining_missing[remaining_missing > 0])

Total Missing Values: 0

Columns with Missing Values:
Series([], dtype: int64)


- Missing `victim_gender` values are assigned `Unknown` because an unavailable gender should not be assumed to belong to the most frequent category.
- `property_loss_usd` is first converted to numeric format. Negative property-loss values are considered logically invalid and converted to missing values. Missing and invalid values are then imputed using the median because monetary data can be influenced by unusually large losses.

### 5.4 Handling Remaining Missing Values

After the initial missing-value treatments, two columns still contain missing values: `victim_gender` and `property_loss_usd`.

Missing `victim_gender` values are represented as `Unknown` because the actual gender cannot be reliably inferred from other records.

The `property_loss_usd` column requires numeric conversion and validation before imputation because monetary values must be numeric and negative property-loss values are logically invalid.

In [30]:
# Handle missing victim gender
df['victim_gender'] = df['victim_gender'].fillna('Unknown')

print("Missing victim_gender values:", df['victim_gender'].isnull().sum())

Missing victim_gender values: 0


### 5.5 Property Loss Validation

The `property_loss_usd` column is converted from object type to numeric format. Values that cannot be converted are treated as missing.

Negative property-loss values are considered logically invalid because financial loss cannot be negative. These values are therefore converted to missing values before imputation.

In [31]:
# Convert property loss to numeric
df['property_loss_usd'] = pd.to_numeric(
    df['property_loss_usd'],
    errors='coerce'
)

# Count missing and invalid negative values
print("Data Type:", df['property_loss_usd'].dtype)
print("Missing Values:", df['property_loss_usd'].isnull().sum())
print("Negative Property Loss Values:", (df['property_loss_usd'] < 0).sum())

Data Type: float64
Missing Values: 0
Negative Property Loss Values: 0


## 6. Data Standardisation

Inconsistent representations of categorical values are standardised to ensure that equivalent values are treated as the same category.

Standardisation includes correcting capitalization, abbreviations, spelling variations, Boolean representations, and inconsistent categorical labels.

In [32]:
# Standardise suspect and victim gender values
gender_mapping = {
    'm': 'Male',
    'male': 'Male',
    'f': 'Female',
    'female': 'Female',
    'unknown': 'Unknown',
    'other': 'Other'
}

for column in ['suspect_gender', 'victim_gender']:
    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(gender_mapping)
        .fillna('Unknown')
    )

print("Suspect Gender:")
print(df['suspect_gender'].value_counts())

print("\nVictim Gender:")
print(df['victim_gender'].value_counts())

Suspect Gender:
suspect_gender
Unknown    1549
Female     1476
Male       1409
Other       287
Name: count, dtype: int64

Victim Gender:
victim_gender
Male       1646
Female     1507
Unknown    1240
Other       328
Name: count, dtype: int64


### 6.1 Standardising Boolean Values

The `reported_online` column contains multiple representations of the same Boolean information, including `Yes`, `YES`, `True`, `1`, `No`, `NO`, `False`, and `0`.

These values are standardised into two consistent categories: `Yes` and `No`.

In [33]:
# Standardise reported_online values
reported_online_mapping = {
    'yes': 'Yes',
    'true': 'Yes',
    '1': 'Yes',
    'no': 'No',
    'false': 'No',
    '0': 'No'
}

df['reported_online'] = (
    df['reported_online']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(reported_online_mapping)
)

print(df['reported_online'].value_counts(dropna=False))

reported_online
No     2606
Yes    2115
Name: count, dtype: int64


### 6.2 Standardising Severity Levels

The `severity` column contains both numeric codes and textual representations of severity levels. Equivalent values are mapped to four consistent categories:

- `1`, `Low`, `low` → `Low`
- `2`, `Med`, `Medium`, `MEDIUM` → `Medium`
- `3`, `High`, `high` → `High`
- `4`, `Crit`, `Critical`, `CRITICAL` → `Critical`

This standardisation ensures that each severity level has a single consistent representation.

In [34]:
# Standardise severity values
severity_mapping = {
    '1': 'Low',
    'low': 'Low',
    '2': 'Medium',
    'med': 'Medium',
    'medium': 'Medium',
    '3': 'High',
    'high': 'High',
    '4': 'Critical',
    'crit': 'Critical',
    'critical': 'Critical'
}

df['severity'] = (
    df['severity']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(severity_mapping)
)

print(df['severity'].value_counts(dropna=False))

severity
Critical    1572
Medium      1268
Low          957
High         924
Name: count, dtype: int64


### 6.3 Standardising Case Status

The `case_status` column contains inconsistent capitalization and spelling errors. Equivalent values are mapped to consistent categories.

Examples include:

- `Open`, `open`, `OPEN` → `Open`
- `Closed`, `closed`, `CLOSED` → `Closed`
- `Pendng` → `Pending`
- `Investgation` and `under investigation` → `Under Investigation`
- `Resolved` → `Resolved`

This ensures that each case status is represented consistently.

In [35]:
# Standardise case status values
case_status_mapping = {
    'open': 'Open',
    'closed': 'Closed',
    'pending': 'Pending',
    'pendng': 'Pending',
    'resolved': 'Resolved',
    'under investigation': 'Under Investigation',
    'investgation': 'Under Investigation'
}

df['case_status'] = (
    df['case_status']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(case_status_mapping)
)

print(df['case_status'].value_counts(dropna=False))

case_status
Closed                 1673
Open                   1030
Under Investigation     995
Pending                 670
Resolved                353
Name: count, dtype: int64


### 6.4 Standardising Resolution

The `resolution` column contains capitalization differences, spelling errors, and multiple labels representing the same outcome.

Equivalent values are consolidated into consistent categories:

- `Arrest Made`, `arrest made`, `Arres Made` → `Arrest Made`
- `No Arrest`, `NO ARREST` → `No Arrest`
- `warning`, `Warning Issued` → `Warning Issued`
- `Dismissed`, `Case Dismissed` → `Case Dismissed`

This reduces duplicate category representations and improves consistency for analysis.

In [36]:
# Standardise resolution values
resolution_mapping = {
    'arrest made': 'Arrest Made',
    'arres made': 'Arrest Made',
    'no arrest': 'No Arrest',
    'warning': 'Warning Issued',
    'warning issued': 'Warning Issued',
    'dismissed': 'Case Dismissed',
    'case dismissed': 'Case Dismissed'
}

df['resolution'] = (
    df['resolution']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(resolution_mapping)
)

print(df['resolution'].value_counts(dropna=False))

resolution
Arrest Made       2168
Warning Issued     874
No Arrest          841
Case Dismissed     838
Name: count, dtype: int64


### 6.5 Standardising Weapon Used

The `weapon_used` column contains inconsistent capitalization and multiple representations of similar weapon categories.

Values are standardised to consistent labels while preserving meaningful distinctions between weapon types.

In [37]:
# Standardise basic formatting in weapon_used
df['weapon_used'] = (
    df['weapon_used']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'knife': 'Knife',
        'blunt object': 'Blunt Object',
        'firearm': 'Firearm',
        'gun': 'Firearm',
        'pistol': 'Firearm',
        'rifle': 'Firearm',
        'hands': 'Hands/Feet',
        'hands/feet': 'Hands/Feet',
        'bat': 'Blunt Object',
        'unarmed': 'Unarmed'
    })
)

print(df['weapon_used'].value_counts(dropna=False))

weapon_used
Hands/Feet      1491
Firearm         1428
Blunt Object     868
Knife            613
Unarmed          321
Name: count, dtype: int64


### 6.6 Standardising District

The `district` column contains inconsistent capitalization, abbreviations, extra whitespace, and alternative representations of the same geographic district.

These variations are consolidated into consistent district categories such as `North`, `South`, `East`, `West`, `Central`, `Northeast`, `Northwest`, `Southeast`, and `Southwest`.

In [38]:
# Standardise district values
def standardize_district(value):
    value = str(value).strip().lower()

    district_mapping = {
        'nor': 'North',
        'north': 'North',
        'sou': 'South',
        'south': 'South',
        'eas': 'East',
        'east': 'East',
        'wes': 'West',
        'west': 'West',
        'cen': 'Central',
        'central': 'Central',
        'mid': 'Central',
        'northeast': 'Northeast',
        'northwest': 'Northwest',
        'southeast': 'Southeast',
        'southwest': 'Southwest'
    }

    return district_mapping.get(value, value.title())

df['district'] = df['district'].apply(standardize_district)

print(df['district'].value_counts())
print("\nNumber of unique districts:", df['district'].nunique())

district
North        656
South        623
Central      569
West         487
East         430
Southwest    420
Northeast    393
Midtown      391
Northwest    379
Southeast    373
Name: count, dtype: int64

Number of unique districts: 10


### 6.7 Inspecting Crime Type Categories

The `crime_type` column contains a large number of inconsistent representations caused by capitalization differences, abbreviations, spelling errors, and extra whitespace.

Before applying standardisation rules, the normalized unique values are inspected to ensure that only genuinely equivalent crime categories are consolidated.

In [39]:
# Inspect normalized crime type values before standardisation
crime_types_normalized = (
    df['crime_type']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
)

print(sorted(crime_types_normalized.unique()))
print("\nNumber of normalized unique crime types:",
      crime_types_normalized.nunique())

['abduction', 'armed robbery', 'arsen', 'arson', 'assault', 'assault & battery', 'asslt', 'b&e', 'battery', 'breaking & entering', 'burglary', 'burglry', 'cyber crime', 'cybercrime', 'd.u.i.', 'deception', 'dom. violence', 'domestc violence', 'domestic violence', 'drug offence', 'drug offense', 'drugs', 'drunk driving', 'dui', 'duii', 'dv', 'dwi', 'fire setting', 'fraud', 'fraudulent activity', 'graffiti', 'hacking', 'homicide', 'homocide', 'kidnaping', 'kidnapping', 'larceny', 'manslaughter', 'murder', 'narcotics', 'online fraud', 'property damage', 'robbery', 'robbry', 'roberry', 'sa', 'scam', 'sex assault', 'sexual assault', 'sexual assualt', 'stealing', 'theft', 'theft/larceny', 'trespass', 'trespassing', 'tresspassing', 'vandalism', 'vandlism']

Number of normalized unique crime types: 58


### 6.8 Standardising Crime Type

The `crime_type` column contained 58 normalized representations due to spelling errors, abbreviations, synonyms, and inconsistent terminology.

Equivalent labels are consolidated into broader, consistent crime categories. For example, `asslt` is standardised to `Assault`, `burglry` to `Burglary`, `DV` to `Domestic Violence`, and `homocide` to `Homicide`.

This reduces category fragmentation while preserving the underlying meaning of each incident.

In [40]:
# Standardise crime type values
crime_mapping = {
    'abduction': 'Kidnapping',
    'kidnaping': 'Kidnapping',
    'kidnapping': 'Kidnapping',

    'arsen': 'Arson',
    'arson': 'Arson',
    'fire setting': 'Arson',

    'assault': 'Assault',
    'asslt': 'Assault',
    'assault & battery': 'Assault',
    'battery': 'Assault',

    'b&e': 'Burglary',
    'breaking & entering': 'Burglary',
    'burglary': 'Burglary',
    'burglry': 'Burglary',

    'cyber crime': 'Cyber Crime',
    'cybercrime': 'Cyber Crime',
    'hacking': 'Cyber Crime',

    'd.u.i.': 'DUI',
    'drunk driving': 'DUI',
    'dui': 'DUI',
    'duii': 'DUI',
    'dwi': 'DUI',

    'dom. violence': 'Domestic Violence',
    'domestc violence': 'Domestic Violence',
    'domestic violence': 'Domestic Violence',
    'dv': 'Domestic Violence',

    'drug offence': 'Drug Offense',
    'drug offense': 'Drug Offense',
    'drugs': 'Drug Offense',
    'narcotics': 'Drug Offense',

    'deception': 'Fraud',
    'fraud': 'Fraud',
    'fraudulent activity': 'Fraud',
    'online fraud': 'Fraud',
    'scam': 'Fraud',

    'homicide': 'Homicide',
    'homocide': 'Homicide',
    'manslaughter': 'Homicide',
    'murder': 'Homicide',

    'armed robbery': 'Robbery',
    'robbery': 'Robbery',
    'robbry': 'Robbery',
    'roberry': 'Robbery',

    'sa': 'Sexual Assault',
    'sex assault': 'Sexual Assault',
    'sexual assault': 'Sexual Assault',
    'sexual assualt': 'Sexual Assault',

    'larceny': 'Theft',
    'stealing': 'Theft',
    'theft': 'Theft',
    'theft/larceny': 'Theft',

    'trespass': 'Trespassing',
    'trespassing': 'Trespassing',
    'tresspassing': 'Trespassing',

    'graffiti': 'Vandalism',
    'vandalism': 'Vandalism',
    'vandlism': 'Vandalism',

    'property damage': 'Property Damage'
}

normalized_crime = (
    df['crime_type']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
)

df['crime_type'] = normalized_crime.map(crime_mapping)

print(df['crime_type'].value_counts(dropna=False))
print("\nNumber of unique crime types:", df['crime_type'].nunique())

crime_type
Fraud                415
DUI                  340
Drug Offense         331
Arson                329
Homicide             316
Theft                314
Kidnapping           311
Sexual Assault       310
Robbery              304
Domestic Violence    294
Trespassing          289
Cyber Crime          289
Burglary             288
Assault              286
Vandalism            263
Property Damage       42
Name: count, dtype: int64

Number of unique crime types: 16


### 6.9 Standardising Incident Date and Time

The `incident_datetime` column contains dates in inconsistent formats and is currently stored as text (`object`).

The values are converted into Pandas `datetime` format using mixed-format parsing. Invalid or unparseable dates are converted to missing values so they can be identified and handled appropriately.

Converting this column to a proper datetime data type enables reliable chronological analysis.

In [41]:
# Convert mixed date formats to datetime
df['incident_datetime'] = pd.to_datetime(
    df['incident_datetime'],
    format='mixed',
    errors='coerce'
)

print("Data Type:", df['incident_datetime'].dtype)
print("Unparseable Dates:", df['incident_datetime'].isnull().sum())
print("\nSample Dates:")
print(df['incident_datetime'].head())

Data Type: datetime64[ns]
Unparseable Dates: 0

Sample Dates:
0   2024-04-16 08:45:03
1   2022-01-11 22:03:29
2   2022-04-14 11:39:24
3   2020-12-09 15:14:24
4   2021-07-24 20:08:13
Name: incident_datetime, dtype: datetime64[ns]


## 7. Data Type Correction

Columns are converted to data types that correctly represent their meaning.

Identifiers such as incident IDs, officer IDs, suspect IDs, victim IDs, and badge numbers are treated as strings because they represent labels rather than quantities. Monetary values are stored as floating-point numbers, while incident dates are stored using the datetime data type.

In [42]:
# Correct data types for identifier columns
id_columns = [
    'incident_id',
    'officer_id',
    'suspect_id',
    'victim_id',
    'badge_number'
]

for column in id_columns:
    df[column] = df[column].astype(str)

# Display corrected data types
print(df[
    id_columns + ['incident_datetime', 'property_loss_usd']
].dtypes)

incident_id                  object
officer_id                   object
suspect_id                   object
victim_id                    object
badge_number                 object
incident_datetime    datetime64[ns]
property_loss_usd           float64
dtype: object


## 8. Outlier Detection Using the IQR Method

Potential outliers in numerical features are identified using the Interquartile Range (IQR) method.

For each selected numerical column, the first quartile (Q1) and third quartile (Q3) are calculated. The IQR is defined as the difference between Q3 and Q1. Values below `Q1 - 1.5 × IQR` or above `Q3 + 1.5 × IQR` are flagged as potential outliers.

Outliers are initially detected rather than automatically removed because extreme values may still represent valid crime incidents. Treatment decisions are made after reviewing the results and considering the meaning of each feature.

In [43]:
# Detect outliers using the IQR method
numeric_columns = [
    'latitude',
    'longitude',
    'suspect_age',
    'victim_age',
    'num_arrests',
    'property_loss_usd'
]

outlier_results = []

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ).sum()

    outlier_results.append([
        column,
        round(lower_bound, 2),
        round(upper_bound, 2),
        outlier_count
    ])

outlier_report = pd.DataFrame(
    outlier_results,
    columns=['Column', 'Lower_Bound', 'Upper_Bound', 'Outlier_Count']
)

outlier_report

,Column,Lower_Bound,Upper_Bound,Outlier_Count
0,latitude,15.67,57.32,0
1,longitude,-145.49,-43.36,181
2,suspect_age,7.50,83.50,0
3,victim_age,-14.00,114.00,0
4,num_arrests,-6.00,10.00,0
5,property_loss_usd,-16078.05,65617.95,0


### 8.1 Outlier Treatment Decision

The IQR analysis detected 181 potential outliers in the `longitude` column, while no outliers were detected in `latitude`, `suspect_age`, `victim_age`, `num_arrests`, or `property_loss_usd`.

The longitude values are retained rather than removed or capped. Although they fall outside the statistical IQR boundaries, they remain within the geographically valid longitude range of -180 to 180 degrees. Therefore, these observations may represent legitimate incident locations rather than data errors.

Earlier, logically impossible values such as invalid latitudes, unrealistic ages, negative arrest counts, and negative property-loss values were treated separately using domain-based validation.

This distinction prevents valid extreme observations from being incorrectly removed solely because they are statistically unusual.

In [44]:
# Final missing-value and duplicate verification
print("Dataset Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

Dataset Shape: (4721, 33)
Total Missing Values: 0
Duplicate Rows: 0


## 9. Before vs. After Data Quality Summary

A final comparison is performed between the original dataset and the cleaned dataset to evaluate the effectiveness of the data-cleaning process.

The comparison includes total row count, missing values, duplicate records, and data type accuracy. Data type accuracy represents the number of columns that have appropriate data types for their intended analytical purpose.

In [45]:
# Before vs. After data quality summary

before_correct_dtypes = 29   # 33 total columns - 4 identified dtype issues
after_correct_dtypes = 33

summary = pd.DataFrame({
    'Metric': [
        'Row Count',
        'Total Missing Values',
        'Duplicate Rows',
        'Correct Data Types'
    ],
    'Before Cleaning': [
        len(df_original),
        df_original.isnull().sum().sum(),
        df_original.duplicated().sum(),
        before_correct_dtypes
    ],
    'After Cleaning': [
        len(df),
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        after_correct_dtypes
    ]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Row Count,5250,4721
1,Total Missing Values,16478,0
2,Duplicate Rows,200,0
3,Correct Data Types,29,33


## 10. Export Cleaned Dataset

After completing all cleaning, standardisation, validation, outlier assessment, and data type corrections, the final analysis-ready dataset is exported to a new CSV file.

The original messy dataset remains unchanged, while the cleaned dataset is saved separately for future analysis.

In [46]:
# Save the cleaned dataset
df.to_csv("crime_incidents_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")
print("Final Shape:", df.shape)

Cleaned dataset saved successfully.
Final Shape: (4721, 33)


## 11. Conclusion

The deliberately messy crime incidents dataset was systematically transformed into a clean and analysis-ready dataset.

The cleaning process included:

- Assessing missing values, duplicates, data types, and value-range anomalies.
- Removing 200 exact duplicate records.
- Handling missing values using appropriate strategies based on the meaning and type of each feature.
- Correcting logically invalid values such as unrealistic ages, invalid latitude values, negative arrest counts, and negative property-loss values.
- Standardising inconsistent categorical representations across gender, crime type, district, severity, case status, resolution, weapon type, and online reporting status.
- Converting incident dates to a consistent datetime format and correcting inappropriate data types.
- Applying the IQR method to detect statistical outliers and retaining valid longitude observations after domain-based evaluation.
- Validating the cleaned dataset to confirm that no missing values or exact duplicate records remained.

The original dataset contained 5,250 rows, while the final cleaned dataset contains 4,721 rows and 33 columns. The cleaned dataset has zero missing values and zero exact duplicate records and has been exported as `crime_incidents_cleaned.csv` for future analysis.